# 01_silver_personal - PySpark Data Cleansing & Star Schema Modeling
Notebook này giúp bạn chạy và kiểm thử từng bước (step-by-step) việc làm sạch dữ liệu thô Bronze và chuyển đổi sang mô hình Star Schema ở tầng Silver.

In [ ]:
# 1. Khởi tạo PySpark Session tại Local
import os
import sys
import glob
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Khắc phục môi trường SSL & Encoding Windows
for env_var in ["REQUESTS_CA_BUNDLE", "CURL_CA_BUNDLE", "SSL_CERT_FILE"]:
    if env_var in os.environ and not os.path.exists(os.environ[env_var]):
        os.environ.pop(env_var, None)

spark = SparkSession.builder \
    .appName("Spotify_Silver_Local") \
    .master("local[*]") \
    .getOrCreate()

print("✅ Khởi tạo PySpark Local Session thành công!")

In [ ]:
# 2. Đọc file dữ liệu Bronze JSON thô từ local (trong thư mục data/bronze_spotify_raw)
local_pattern = os.path.abspath("data/bronze_spotify_raw/personal_*.json").replace("\\", "/")
json_files = [f.replace("\\", "/") for f in glob.glob(local_pattern)]

if json_files:
    print(f"📥 Đã tìm thấy {len(json_files)} file JSON thô local: {json_files[0]}")
    df_raw = spark.read.option("multiline", "true").json(json_files)
    df_raw.printSchema()
else:
    print("⚠️ Chưa tìm thấy file JSON thô trong data/bronze_spotify_raw/")

In [ ]:
# 3. Unnest mảng JSON 'items' chứa thông tin các bài hát vừa nghe
df_exploded = df_raw.select(
    F.explode("items").alias("item"),
    F.col("ingestion_metadata.ingestion_time").alias("ingestion_time")
)
print(f"📊 Tổng số dòng sau khi explode: {df_exploded.count()}")
df_exploded.select("item.played_at", "item.track.name").show(5, truncate=False)

In [ ]:
# 4. Trích xuất các trường dữ liệu phẳng (Flattening fields)
df_flattened = df_exploded.select(
    F.to_timestamp(F.col("item.played_at")).alias("played_at"),
    F.col("item.track.id").alias("track_id"),
    F.col("item.track.name").alias("track_name"),
    F.col("item.track.duration_ms").cast("integer").alias("duration_ms"),
    F.col("item.track.explicit").cast("boolean").alias("explicit"),
    F.col("item.track.album.id").alias("album_id"),
    F.col("item.track.album.name").alias("album_name"),
    F.explode("item.track.artists").alias("artist"),
    F.col("ingestion_time")
).select(
    "played_at",
    "track_id",
    "track_name",
    "duration_ms",
    "explicit",
    "album_id",
    "album_name",
    F.col("artist.id").alias("artist_id"),
    F.col("artist.name").alias("artist_name"),
    "ingestion_time"
)

print("✅ Đã làm sạch & phẳng hóa dữ liệu:")
df_flattened.show(5, truncate=False)

In [ ]:
# 5. Kiểm thử tạo dữ liệu bảng Dim Tracks
dim_tracks_df = df_flattened.select(
    "track_id", "track_name", "duration_ms", "explicit", "album_id"
).dropDuplicates(["track_id"])

print(f"🎵 Bảng dim_tracks có {dim_tracks_df.count()} bài hát độc bản:")
dim_tracks_df.show(5, truncate=False)

In [ ]:
# 6. Kiểm thử tạo dữ liệu bảng Dim Artists
dim_artists_df = df_flattened.select("artist_id", "artist_name").dropDuplicates(["artist_id"])

print(f"🎤 Bảng dim_artists có {dim_artists_df.count()} nghệ sĩ độc bản:")
dim_artists_df.show(5, truncate=False)

In [ ]:
# 7. Kiểm thử tạo dữ liệu bảng Fact Streams
fact_streams_df = df_flattened.select(
    "played_at", "track_id", "artist_id", "album_id", "ingestion_time"
).dropDuplicates(["played_at", "track_id"])

print(f"🎧 Bảng fact_streams có {fact_streams_df.count()} lượt nghe độc bản:")
fact_streams_df.show(5, truncate=False)